# Bimanual Planning Example Notebook

This notebook demonstrates the complete workflow for planning constrained bimanual motions using the minimal coordinates strategy.

In [66]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [67]:
# Fix relative paths so contents of the src directory can be imported.
import sys
sys.path.append("..")
sys.path.append("../../constrained-bimanual-planning-example")

In [68]:
import numpy as np
import os
import time
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import networkx as nx
import torch

In [69]:
from pydrake.all import (
    StartMeshcat,
    CollisionCheckerParams,
    RobotDiagramBuilder,
    MeshcatVisualizerParams,
    Role,
    MeshcatVisualizer,
    Parser,
    LoadModelDirectives,
    ProcessModelDirectives,
    SceneGraphCollisionChecker,
    AutoDiffXd,
    RigidTransform_,
    IrisNp2Options,
    IrisZoOptions,
    SnoptSolver,
    IpoptSolver,
    IrisParameterizationFunction,
    MathematicalProgram,
    HPolyhedron,
    IrisNp2,
    IrisZo,
    Hyperellipsoid,
    RandomGenerator,
    ComputePairwiseIntersections,
    GcsTrajectoryOptimization,
    Point,
    GraphOfConvexSetsOptions,
    FunctionHandleTrajectory,
    InitializeAutoDiff,
    ExtractGradient,
    Toppra,
    PathParameterizedTrajectory,
    PiecewisePolynomial,
    CompositeTrajectory,
    CalcGridPointsOptions,
    BsplineBasis,
    BsplineTrajectory,
    KinematicTrajectoryOptimization,
    MinimumDistanceLowerBoundConstraint,
    PyFunctionConstraint,
    SolverOptions,
    CommonSolverOption,
    Solve,
    sqrt,
    DiagramBuilder,
    TrajectorySource,
    InverseDynamicsController,
    Demultiplexer,
)

In [70]:
# We replace the analytic_ik with the NN
import src.iiwa_analytic_ik as iiwa_analytic_ik
import src.common as common
import src.rrt as rrt
import src.shortcut as shortcut
import src.utils as utils
from src.iiwa_program import Iiwa14IKProgram
from ikflow.config import DEVICE


The meshcat visualization can be viewed in your browser, by opening the link that appears after running the following cell.

In [71]:
# Only run this cell once.
meshcat = StartMeshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7001


# Parameters

- `directives_file` is a scene description, giving the location of all the models in the world.
- `grasp_distance` is the distance between the end-effectors of the two robot arms.
- `GC2`, `GC4`, and `GC6` are the "global configuration parameters", describing which branch of the IK function to use.
- `q_tilde_bottom`, `q_tilde_middle`, and `q_tilde_top` are three key configurations that we plan between, represented in the parameterized coordinates.
- `seeds` is a list of configurations to use as seed points for the region generation. They were chosen by hand, using a virtual teleoperation notebook, similar to [this one](https://deepnote.com/workspace/Manipulation-ac8201a1-470a-4c77-afd0-2cc45bc229ff/project/0762b167-402a-4362-9702-7d559f0e73bb/notebook/iris_builder-3c25c10bc29d4c9493e48eaced475d03). One can also generate regions to cover a graph in configuration space (shown later in the code for RRT) or use the [clique covers approach](https://ieeexplore.ieee.org/abstract/document/10610005/).

In [72]:
directives_file = os.path.join(common.RepoDir(), "models/old_shelves.dmd.yaml")
grasp_distance = 0.6
q_tilde_bottom = np.array([-0.6430910102907225, 1.9156121024586796, -1.7968254667817805, 1.2945447141185198, -0.023834531305537934, -0.876966810663043, -1.7041643160834519, 0.73227135, -0.64257539, -0.17809318, -0.57395456, -0.20437532, -0.4864951,
 -0.18577532, -0.38053642])
q_tilde_middle = np.array([-0.5997312520566763, 1.489780849654964, -1.4739679827359913, 1.2905366081785483, -0.04421061906813227, -0.8793712572715165, -1.1603461715511334, 1.45])
q_tilde_top = np.array([-0.1994994216078726, 0.9140739951190965, -2.236618320862171, 0.5238879195899456, 0.7998441913611017, -1.3575398006936048, -1.0153092816310436, -0.1724197671774095, 0.09183837492938327, 1.0675584570477854, -1.0616344482718332, 0.21734820797221702, 0.11781950435342614, -1.68411089296333, -1.1857552653505647])

In [73]:
seeds = [
    q_tilde_bottom,
    np.array([-0.7341522021700233, 1.9192492722970935, -1.849050540687353, 1.4690188979347225, -0.022913995470214974, -0.7839567180379224, -1.735834076048031, 1.45]),
    np.array([-0.816394667473979, 1.9228828117510568, -1.9042766014076622, 1.6254903325102958, -0.020884458583263387, -0.6994788210824544, -1.773950224396859, 1.45]),
    np.array([-0.9076736984240236, 1.7999568628541147, -1.8278258357789336, 1.8976493299850326, -0.032028511314404574, -0.5492230012012871, -1.624933169711267, 1.45]),
    np.array([-0.90780384835653, 1.5443282072400564, -1.480882097408486, 1.9741581801564516, -0.07059018895327443, -0.5065618808846135, -1.1610690777465094, 1.45]),
    np.array([-0.877792385473089, 1.283945692440691, -1.1673903163525974, 1.7986279782674526, -0.08798686286997325, -0.605914842625335, -0.7496023024205761, 1.45]),
    np.array([-0.7363360141869535, 1.0835790623705088, -1.102219288049605, 1.3727471630916555, -0.07210415656873102, -0.8362237374759414, -0.6008766712030682, 1.45]),
    np.array([-0.7093225760311644, 0.8650840325295542, -1.4794100092984808, 1.2099934253928784, 0.44173726212402287, -0.9673197772349095, -0.9450827150678346, 2.0]),
    np.array([-0.5237049267440886, 0.7086764066165658, -1.9872212610757156, 1.045742737284787, 0.8594286107005795, -1.171705603794283, -1.1435157398017397, 2.41]),
    np.array([-0.37540312953312194, 0.7958305227244739, -2.112215906760149, 0.8433434932970723, 0.8316630398644385, -1.2430896040746857, -1.1077155278001196, 2.41]),
    q_tilde_top,
    np.array([-0.7686406052800139, 1.504938625148829, -1.4584578152597332, 1.655937158932382, -0.055175677810583384, -0.6834840454669682, -1.1418310479792013, 1.45]),
    q_tilde_middle,
]

# Set Up Environment

This is general, boilerplate code that most Drake projects include. Note the usage of [`RobotDiagramBuilder`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1planning_1_1_robot_diagram_builder.html) to construct a [`RobotDiagram`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1planning_1_1_robot_diagram.html) and [`CollisionChecker`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1planning_1_1_collision_checker.html), as opposed to the less specific [`DiagramBuilder`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1systems_1_1_diagram_builder.html). We specifically use a [`SceneGraphCollisionChecker`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1planning_1_1_scene_graph_collision_checker.html).

In [74]:
params = CollisionCheckerParams()
builder = RobotDiagramBuilder(time_step=0.0)

meshcat_visual_params = MeshcatVisualizerParams()
meshcat_visual_params.delete_on_initialization_event = False
meshcat_visual_params.role = Role.kIllustration
meshcat_visual_params.prefix = "visual"
meshcat_visual = MeshcatVisualizer.AddToBuilder(
    builder.builder(), builder.scene_graph(), meshcat, meshcat_visual_params)

meshcat_collision_params = MeshcatVisualizerParams()
meshcat_collision_params.delete_on_initialization_event = False
meshcat_collision_params.role = Role.kProximity
meshcat_collision_params.prefix = "collision"
meshcat_collision_params.visible_by_default = False
meshcat_collision = MeshcatVisualizer.AddToBuilder(
    builder.builder(), builder.scene_graph(), meshcat, meshcat_collision_params)

plant = builder.plant()
parser = Parser(plant)
package_xml_path = os.path.join(common.RepoDir(), "package.xml")
parser.package_map().AddPackageXml(package_xml_path)
directives = LoadModelDirectives(directives_file)
ProcessModelDirectives(directives, parser)

params.robot_model_instances = [
    plant.GetModelInstanceByName("iiwa_left"),
    plant.GetModelInstanceByName("iiwa_right")
]

plant.Finalize()

# We export these inputs and outputs so we can wrap the RobotDiagram in a larger
# Diagram, which will include a controller, to simulate and visualize.
builder.builder().ExportInput(plant.get_actuation_input_port(), "actuation")
builder.builder().ExportOutput(plant.get_state_output_port(), "state")

diagram = builder.Build()

params.model = diagram
params.edge_step_size = 0.01
checker = SceneGraphCollisionChecker(params)

context = diagram.CreateDefaultContext()
plant_context = plant.GetMyContextFromRoot(context)
diagram.ForcedPublish(context)

INFO:drake:Allocating contexts to support implicit context parallelism 8


# Build Regions

## Set Up the Parameterization

Check out `src/iiwa_analytic_ik.py` for more details on the implementation of the analytic IK function itself. The key special aspect needed for this project is making it compatible with both `float` and Drake's `AutoDiffXd` scalar type (or numpy arrays of each). When called with `AutoDiffXd`, it will automatically perform forward-mode automatic differentiation. Care must be taken to return objects of the correct template type -- see [this documentation](https://drake.mit.edu/python_bindings.html#c-function-and-method-template-instantiations-in-python) for more information on how Drake handles templating in Python.

The parameterization has two parts, the callable function itself, and the [`IrisParameterizationFunction`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1planning_1_1_iris_parameterization_function.html) object, which wraps the callable and maintains some additional necessary information (input dimension and thread safety). The parameterization itself takes as input the configuration of the controlled arm and the self motion parameter of the subordinate arm, and outputs the configuration of the controlled arm concatenated with the configuration of the follower arm.

In [ ]:
program_right = Iiwa14IKProgram(diagram, model_instance=plant.GetModelInstanceByName("iiwa_right"))
program_left = Iiwa14IKProgram(diagram, model_instance=plant.GetModelInstanceByName("iiwa_left"))


def q_to_ee_target(q):
    """
    Given leader (left) arm configuration, compute target pose for follower (right) link_7.
    
    Input: q - 7 joint angles for left arm (leader)
    Output: RigidTransform - target pose of right link_7, relative to right base frame
    """
    global grasp_distance
    
    ad = isinstance(q[0], AutoDiffXd) if len(np.atleast_1d(q)) > 0 else False
    T = AutoDiffXd if ad else float
    q_full = np.zeros(14, dtype=type(q[0]))
    q_full[:7] = q 
    T_left_link7_world = program_left.fk(q_full, matrix=True)
    R_left = T_left_link7_world[:-1, :-1]
    p_left = T_left_link7_world[:-1, -1]
    ang = (180 - 2. * 68.) * np.pi / 180. 
    c, s = np.cos(ang), np.sin(ang)
    R_adjusted = R_left @ np.array([[-1, 0, 0], [0, 1, 0], [0, 0, -1]]) @ np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])
    p_target = p_left + R_adjusted @ np.array([0, 0, -grasp_distance])
    T_target_world = np.eye(4, dtype=type(q[0]))
    T_target_world[:-1, :-1] = R_adjusted
    T_target_world[:-1, -1] = p_target
    T_right_base_world = np.eye(4, dtype=type(q[0]))
    T_right_base_world[:-1, -1] = np.array([0, 0.765, 0])
    T_target_relative = np.linalg.inv(T_right_base_world) @ T_target_world


    # utils.DrawAxes(RigidTransform_[float](T_target_relative), meshcat, name="t_target_relative")
    # utils.DrawAxes(RigidTransform_[float](T_target_world), meshcat, name="t_target_world")
    
    return RigidTransform_[T](T_target_relative), RigidTransform_[T](T_target_world)

def parameterization(q_tilde):
    '''q_tilde: 
    input: [7 joints of left (leader) + 8 latent variables for right (follower) IK]
    output: [7 left joints, 7 right joints] (14 total)
    '''
    q_full = np.zeros(14, dtype=type(q_tilde[0]))
    
    # Left arm (leader) configuration from input
    q_left = q_tilde[:7]
    
    # Compute target for right arm (follower)
    tf_goal, _ = q_to_ee_target(q_left)

    # Solve for right arm (follower) using NN
    conditional = [tf_goal.translation(), tf_goal.rotation().ToQuaternion().wxyz()]
    latent = q_tilde[7:]
    q_right = program_right.ik_inference(vars=np.concatenate(conditional + [latent]), add_correction=False).detach().cpu().numpy()
    
    # Assemble full config: [left_arm_7, right_arm_7]
    q_full[:7] = q_left
    q_full[7:] = q_right
    
    return q_full


WorldModel::LoadRobot: /home/tangles/.cache/jrl/urdfs/iiwa14_formatted_link_filepaths_absolute.urdf
URDFParser: Link size: 11
URDFParser: Joint size: 11
URDFParser: Done loading robot file /home/tangles/.cache/jrl/urdfs/iiwa14_formatted_link_filepaths_absolute.urdf
WorldModel::LoadRobot: /home/tangles/.cache/jrl/urdfs/iiwa14_formatted_link_filepaths_absolute.urdf
URDFParser: Link size: 11
URDFParser: Joint size: 11
URDFParser: Done loading robot file /home/tangles/.cache/jrl/urdfs/iiwa14_formatted_link_filepaths_absolute.urdf


In [ ]:
plant.SetPositions(plant_context, parameterization(q_tilde_bottom))
diagram.ForcedPublish(context)

In [ ]:
plant.SetPositions(plant_context, parameterization(q_tilde_top))
diagram.ForcedPublish(context)

In [ ]:
# idx = 4

# plant.SetPositions(plant_context, parameterization(seeds[idx]))
# diagram.ForcedPublish(context)

## Planning with Bidirectional RRT

I've included in this repository a simple Python implementation of RRT and BiRRT. They rely on two oracles: a random configuration generator, and a validity checker. (The metric is assumed to be the L2 norm.) In the parameterized space, it is easy to generate a random configuration given the domain that we used for IrisNp2 and IrisZo. And we can check validity by simply checking for reachability, subordinate arm joint limit violations, and collisions. We follow this up by running randomized shortcutting (also a simple Python implementation) to improve the path a bit.

Finally, we construct a formal Drake [`Trajectory`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1trajectories_1_1_trajectory.html) object. We construct a twice-differentiable path for each segment with zero initial and final velocity using [`PiecewisePolynomial::CubicWithContinuousSecondDerivatives`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1trajectories_1_1_piecewise_polynomial.html#aba4275b536c162df6d8e2c06b4036f3a), before concatenating them as a [`CompositeTrajectory`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1trajectories_1_1_composite_trajectory.html).

In [185]:
def RandomConfig():
    q_tilde = np.zeros(15)
    q_tilde[:7] = np.random.uniform(low=iiwa_analytic_ik.iiwa_limits_lower, 
                                    high = iiwa_analytic_ik.iiwa_limits_upper)
    q_tilde[7:] = np.random.randn(8)
    return q_tilde

def ValidityChecker(q_tilde):
    q_target, world_target = q_to_ee_target(q_tilde[:7])
    q_full = parameterization(q_tilde)
    
    angle_error, d_error = utils.CalculateError(RigidTransform_[float](program_right.fk(q_full, matrix=True)), 
                                                world_target)
    if angle_error > 0.1: return False
    if d_error > 0.1: return False

    

    if (q_full[7:] < iiwa_analytic_ik.iiwa_limits_lower).any(): return False
    if (q_full[7:] > iiwa_analytic_ik.iiwa_limits_upper).any(): return False


    if not checker.CheckConfigCollisionFree(q_full): return False
    
    
    # c = torch.tensor(np.hstack([q_target.translation(), q_target.rotation().ToQuaternion().wxyz()]), dtype=torch.float32, device=DEVICE)
    # q_torch = torch.tensor(q_full[7:], dtype=torch.float32, device=DEVICE)
    # vars = torch.cat((c, q_torch))

    # jac = program_right.rev_jac_gen(vars)
    # if torch.linalg.svdvals(jac.squeeze(0))[7] < 1.0: return False




    return True


ValidityChecker(q_tilde_top)

True

In [ ]:
rrt_options = rrt.RRTOptions(
    step_size = 2e-2,
    check_size = 5e-3,
    max_vertices = 1e4,
    max_iters = 1e6,
    goal_sample_frequency = 0.01,
    always_swap = False
)
rrt_planner = rrt.BiRRT(RandomConfig, ValidityChecker)

np.random.seed(0)

start = q_tilde_bottom.copy()
goal = q_tilde_top.copy()


path = rrt_planner.plan(start, goal, rrt_options)

Iterations:   0%|          | 0/1000000 [00:00<?, ?it/s]

Vertices:   0%|          | 0/10000 [00:00<?, ?it/s]

In [102]:
# We also run randomized shortcutting to improve the paths at least a little bit.
np.random.seed(0)
shortcut_path = shortcut.shortcut(path.copy(), ValidityChecker, num_tries=1e2, check_size=rrt_options.check_size)

  0%|          | 0/100 [00:00<?, ?it/s]

Applied 13 shortcuts


In [103]:
rrt_traj_segments = [
    PiecewisePolynomial.CubicWithContinuousSecondDerivatives(
        np.array([float(i) - 1.0, float(i)]),
        np.array([shortcut_path[i-1], shortcut_path[i]]).T,
        np.zeros(15),
        np.zeros(15)
    )
    for i in range(1, len(shortcut_path))
]
rrt_traj = CompositeTrajectory(rrt_traj_segments)

In [144]:
# Visualize the trajectory by sampling and animating it
import time

# Sample the trajectory at regular time intervals
t_start = rrt_traj.start_time()
t_end = rrt_traj.end_time()
num_samples = 300
times = np.linspace(t_start, t_end, num_samples)

# Save the sampled trajectory so other notebooks or scripts can load it later.
q_tilde_samples = []
q_full_samples = []

# Animate the trajectory
for t in tqdm(times):
    q_tilde = rrt_traj.value(t).flatten()
    q_full = parameterization(q_tilde)
    q_tilde_samples.append(q_tilde.copy())
    q_full_samples.append(q_full.copy())
    plant.SetPositions(plant_context, q_full)
    diagram.ForcedPublish(context)
    time.sleep(0.01)  # Small delay to make animation viewable
    

q_tilde_samples = np.asarray(q_tilde_samples)
q_full_samples = np.asarray(q_full_samples)
output_dir = os.path.abspath(os.path.join(common.RepoDir(), "../ik-net-optimization/notebooks"))
np.save(os.path.join(output_dir, "rrt_q_tilde_samples.npy"), q_tilde_samples)
np.save(os.path.join(output_dir, "rrt_q_full_samples.npy"), q_full_samples)

print("Trajectory visualization complete!")

  0%|          | 0/300 [00:00<?, ?it/s]

Trajectory visualization complete!


In [180]:
program_right.create_prog()
for t in tqdm(times):
    q_tilde = rrt_traj.value(t).flatten()
    q_target, _ = q_to_ee_target(q_tilde[:7])
    c = torch.tensor(np.hstack([q_target.translation(), q_target.rotation().ToQuaternion().wxyz()]), dtype=torch.float32, device=DEVICE)
    q_full = parameterization(q_tilde)
    q_torch = torch.tensor(q_full[7:], dtype=torch.float32, device=DEVICE)
    vars = torch.cat((c, q_torch))

    jac = program_right.rev_jac_gen(vars)
    print(torch.linalg.svdvals(jac.squeeze(0))[7].detach().cpu().numpy().flatten())

    plant.SetPositions(plant_context, q_full)
    diagram.ForcedPublish(context)

  0%|          | 0/300 [00:00<?, ?it/s]

[532.18646]
[404.638]
[999.3056]
[585.7932]
[472.08673]
[1190.2433]
[966.214]
[983.403]
[322.38538]
[1479.6626]
[804.8598]
[743.02765]
[1360.0736]
[889.0603]
[885.29425]
[485.65887]
[7106.8613]
[2637.734]
[2311.9387]
[1248.0432]
[418.32034]
[1593.4624]
[1443.1656]
[1592.0789]
[947.4958]
[1192.8198]
[1352.0337]
[1578.4707]
[845.9575]
[1586.1232]
[1699.9672]
[844.70197]
[2224.6191]
[779.17957]
[519.9723]
[797.5256]
[1282.8917]
[489.18863]
[3323.6104]
[1855.9183]
[1383.718]
[3512.109]
[213702.55]
[32.786842]
[17.618223]
[27.930197]
[40.411686]
[161.1248]
[103.68619]
[97.29026]
[102.92882]
[84.730576]
[90.98956]
[108.97352]
[227.47421]
[472.048]
[591.25024]
[396.0289]
[210.42372]
[180.73389]
[152.64967]
[482.30865]
[1834.2072]
[128.74615]
[168.72847]
[76.833626]
[204.28613]
[67.40195]
[53.947544]
[7004.652]
[4706.687]
[1637.7534]
[2580.6716]
[1990.1346]
[2094.0151]
[1669.2108]
[3018.1902]
[3249.5266]
[4648.782]
[67.56071]
[158.00995]
[51.320065]
[70.59999]
[70.28763]
[101.44079]
[72.199]
[

In [179]:
for t in tqdm(times):
    q_tilde = rrt_traj.value(t).flatten()
    q_full = parameterization(q_tilde)
    q_full_autodiff = np.full(14, AutoDiffXd(0));
    for i in range(len(q_full)):
        deriv = np.zeros(14)
        deriv[i] = 1
        q_full_autodiff[i] = AutoDiffXd(q_full[i], derivatives=deriv)
    pose = program_right.fk(q_full_autodiff)
    pose = np.hstack([pose[0], pose[1]])
    jac = np.zeros((7, 7))
    for i in range(7):
        jac[i, :] = pose[i].derivatives()[7:]
    
    print(np.linalg.svd(jac, compute_uv=False)[5])
       




  0%|          | 0/300 [00:00<?, ?it/s]

0.18850891278748053
0.18915678030581864
0.1908580205084018
0.19317453986254268
0.19582732109714
0.1985563992881187
0.20086533952261165
0.20252487620975354
0.2033339421565626
0.2036330342903037
0.2046880788527847
0.20631171290386702
0.20816322057085807
0.20986454231332527
0.21126050781406502
0.21227429627008404
0.21280713749907226
0.21286781281866088
0.2125334417301138
0.2119663987245164
0.21133433564478316
0.21062047138821113
0.20984282886396363
0.20914592084191763
0.2086450205150794
0.20847858102583586
0.2083128549207369
0.20788897594251776
0.2072708565691857
0.20652023700187946
0.20581069647787165
0.2052067545589519
0.2047631476891629
0.20452006147349888
0.20429972938799032
0.20325448032273066
0.20140160159990278
0.198398399950305
0.19344315297078035
0.18729466389814586
0.1813750335275109
0.17760158457832434
0.17703580759776907
0.17702530344397222
0.17700742798913693
0.17694779105556466
0.17688859143859478
0.17683539277369048
0.1767847550622092
0.17674521423082676
0.17675731229015065

In [148]:
AutoDiffXd(1, derivatives=[1, 0]).derivatives

<bound method PyCapsule.derivatives of <AutoDiffXd 1.0 nderiv=2>>